In [16]:
!pip install yfinance plotly lxml html5lib -q

In [21]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests

In [28]:
print("Fetching current S&P 500 tickers from Wikipedia...")
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

# Mascheriamo lo script da browser reale per evitare l'errore 403 Forbidden
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response = requests.get(url, headers=headers)

# Passiamo l'HTML testuale a pandas invece dell'URL diretto
table = pd.read_html(response.text)[0]
tickers = table['Symbol'].str.replace('.', '-', regex=False).tolist()
print(f"Retrieved {len(tickers)} tickers.")

start_date = "2000-01-01"
end_date = "2026-02-18"

print("Downloading S&P 500 stock data (this may take 1-2 minutes)...")
data = yf.download(tickers, start=start_date, end=end_date, progress=False)
close_prices = data['Close']
high_prices = data['High']
low_prices = data['Low']

print("Downloading SPY data (Index baseline)...")
spy_data = yf.download("SPY", start=start_date, end=end_date, progress=False)
spy_close = spy_data['Close']
if isinstance(spy_close, pd.DataFrame):
    spy_close = spy_close.iloc[:, 0]

# Aligning indexes to prevent calculation errors across different trading days
valid_dates = spy_close.dropna().index
close_prices = close_prices.reindex(valid_dates).ffill()
high_prices = high_prices.reindex(valid_dates).ffill()
low_prices = low_prices.reindex(valid_dates).ffill()
spy_close = spy_close.reindex(valid_dates).ffill()

# Pre-calculate daily returns for the next blocks
daily_returns = close_prices.pct_change()
print("Data ready for analysis!")

Fetching current S&P 500 tickers from Wikipedia...


/tmp/ipython-input-1192549771.py:11: FutureWarning:

Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.

/tmp/ipython-input-1192549771.py:19: FutureWarning:

YF.download() has changed argument auto_adjust default to True



Retrieved 503 tickers.


/tmp/ipython-input-1192549771.py:25: FutureWarning:

YF.download() has changed argument auto_adjust default to True



Data ready for analysis!


In [29]:
# --- BLOCK 2: METHOD 1 - FIXED THRESHOLD (-7%) ---

# 1. Calculation
fixed_drop_condition = (daily_returns <= -0.07).astype(int)
fixed_drop_count = fixed_drop_condition.sum(axis=1).rolling(window=5).sum()

# 2. Plotting
fig1 = make_subplots(specs=[[{"secondary_y": True}]])
valid_data1 = fixed_drop_count.dropna().index

# Add Traces
fig1.add_trace(go.Scatter(x=valid_data1, y=spy_close.loc[valid_data1], name="SPY Index",
                          line=dict(color='#1f77b4', width=1.5)), secondary_y=False)
fig1.add_trace(go.Scatter(x=valid_data1, y=fixed_drop_count.loc[valid_data1], name="Count -7%",
                          fill='tozeroy', line=dict(color='orange', width=1), opacity=0.7), secondary_y=True)

# Layout Formatting
fig1.update_layout(title_text="Method 1: Fixed -7% Threshold (Rolling 5d)", template="plotly_dark",
                   height=500, hovermode="x unified", margin=dict(l=20, r=20, t=60, b=20))
fig1.update_yaxes(title_text="SPY Level (Log)", type="log", secondary_y=False)
fig1.update_yaxes(title_text="Stocks Count", rangemode="tozero", secondary_y=True)

fig1.show()

In [30]:
# --- BLOCK 3: METHOD 2 - RELATIVE STANDARD DEVIATION (> 3x Std Dev) ---

# 1. Calculation
rolling_std_20d = daily_returns.rolling(window=20).std()
std_drop_condition = (daily_returns <= -3 * rolling_std_20d).astype(int)
std_drop_count = std_drop_condition.sum(axis=1).rolling(window=5).sum()

# 2. Plotting
fig2 = make_subplots(specs=[[{"secondary_y": True}]])
valid_data2 = std_drop_count.dropna().index

# Add Traces
fig2.add_trace(go.Scatter(x=valid_data2, y=spy_close.loc[valid_data2], name="SPY Index",
                          line=dict(color='#1f77b4', width=1.5)), secondary_y=False)
fig2.add_trace(go.Scatter(x=valid_data2, y=std_drop_count.loc[valid_data2], name="Count StdDev",
                          fill='tozeroy', line=dict(color='crimson', width=1), opacity=0.7), secondary_y=True)

# Layout Formatting
fig2.update_layout(title_text="Method 2: Drop > 3x Std Dev (20d)", template="plotly_dark",
                   height=500, hovermode="x unified", margin=dict(l=20, r=20, t=60, b=20))
fig2.update_yaxes(title_text="SPY Level (Log)", type="log", secondary_y=False)
fig2.update_yaxes(title_text="Stocks Count", rangemode="tozero", secondary_y=True)

fig2.show()

In [31]:
# --- BLOCK 4: METHOD 3 - AVERAGE TRUE RANGE (Drop in $ > 3x ATR) ---

# 1. Calculation
prev_close = close_prices.shift(1)
true_range_1 = high_prices - low_prices
true_range_2 = (high_prices - prev_close).abs()
true_range_3 = (low_prices - prev_close).abs()

true_range = pd.DataFrame(np.maximum(np.maximum(true_range_1.values, true_range_2.values), true_range_3.values),
                          index=close_prices.index, columns=close_prices.columns)

atr_14d = true_range.rolling(window=14).mean()
absolute_drop = close_prices - prev_close

# Avoid division by zero
atr_drop_condition = (absolute_drop <= -3 * atr_14d) & (atr_14d > 0)
atr_drop_condition = atr_drop_condition.astype(int)
atr_drop_count = atr_drop_condition.sum(axis=1).rolling(window=5).sum()

# 2. Plotting
fig3 = make_subplots(specs=[[{"secondary_y": True}]])
valid_data3 = atr_drop_count.dropna().index

# Add Traces
fig3.add_trace(go.Scatter(x=valid_data3, y=spy_close.loc[valid_data3], name="SPY Index",
                          line=dict(color='#1f77b4', width=1.5)), secondary_y=False)
fig3.add_trace(go.Scatter(x=valid_data3, y=atr_drop_count.loc[valid_data3], name="Count ATR",
                          fill='tozeroy', line=dict(color='mediumorchid', width=1), opacity=0.7), secondary_y=True)

# Layout Formatting
fig3.update_layout(title_text="Method 3: Drop > 3x ATR (14d)", template="plotly_dark",
                   height=500, hovermode="x unified", margin=dict(l=20, r=20, t=60, b=20))
fig3.update_yaxes(title_text="SPY Level (Log)", type="log", secondary_y=False)
fig3.update_yaxes(title_text="Stocks Count", rangemode="tozero", secondary_y=True)

fig3.show()